# Edge V1 Latency

True batched decode latency of the v1 codec; writes `latency_v1_true.json`.


In [ ]:
import os, time, warnings
os.environ.update(PYTHONWARNINGS='ignore', PYDEVD_DISABLE_FILE_VALIDATION='1', TOKENIZERS_PARALLELISM='false',
                  HF_HUB_DISABLE_PROGRESS_BARS='1', TQDM_DISABLE='1')
warnings.filterwarnings('ignore')
T0 = time.time()

In [ ]:
import os, subprocess
REPO, URL, BRANCH = "/tmp/repo", "https://github.com/saltypal/SemanticSchedulerEnd2End.git", "main"
def _git(*a, cwd=None):
    return subprocess.run(["git", *a], cwd=cwd, check=True, capture_output=True, text=True).stdout.strip()
if os.path.isdir(REPO + "/.git"):
    _git("fetch", "-q", "--depth", "1", "origin", BRANCH, cwd=REPO)
    _git("reset", "-q", "--hard", "FETCH_HEAD", cwd=REPO)
else:
    _git("clone", "--depth", "1", "-q", "--no-progress", "--branch", BRANCH, URL, REPO)
GIT_SHA = _git("rev-parse", "HEAD", cwd=REPO)
print("repo @", GIT_SHA[:8], "(" + _git("log", "-1", "--format=%s", cwd=REPO) + ")")

In [ ]:
import subprocess, time
_t = time.time()
LOG = "/kaggle/working/pip_install.log"
with open(LOG, "w") as _f:
    _rc = subprocess.run(["pip", "install", "-q", "--no-input", "--disable-pip-version-check",
                          "--ignore-requires-python", "-e", ".", *['transformers>=4.49.0', 'PyYAML', 'lpips']],
                         cwd=REPO, stdout=_f, stderr=subprocess.STDOUT).returncode
if _rc:
    print("".join(open(LOG).readlines()[-25:]))
    raise RuntimeError("pip install failed (rc=%s)" % _rc)
print("installed in %.0fs" % (time.time() - _t))

In [ ]:
import json, sys, time, traceback
from pathlib import Path
try:
    import json, sys, time
    sys.path.insert(0, REPO)
    import numpy as np, torch
    from pathlib import Path
    from Codec.edge import feature_pass as fp
    from Codec.edge.feature_pass_backends import RealSQGAN
    from Evaluation.edge.edge_config import load_config
    M, N_IMAGES, WARMUP, SAMPLES = 0.25, 200, 5, 40
    cfg = json.loads(json.dumps(load_config("feature_pass")))
    cfg["sqgan"]["codec"] = "v1"; cfg["sqgan"]["m_levels"] = [M]
    root = fp.find_data_root(cfg["paths"]["images_marker"], cfg["paths"]["input_base"], cfg["paths"]["discover_depth"])
    rows = [json.loads(l) for l in (root / cfg["paths"]["images_marker"]).read_text().splitlines() if l.strip()]
    val = [r for r in rows if r.get("split") == "val"] or rows
    sq = RealSQGAN(cfg, "cuda")
    codec = sq._codec(M)
    payloads = []
    for r in val[:N_IMAGES]:
        arr, _ = fp.letterbox(fp._load_image(root, r), tuple(cfg["sqgan"]["input_hw"]), cfg["sqgan"]["pad_value"])
        _, handle = sq.segment(arr)
        payloads.append(sq.encode(arr, handle, M)["payload"])
    print("encoded", len(payloads), "payloads")
    samples, vram = {}, {}
    for b in (1, 2, 4, 8):
        batches = [[payloads[(k * b + i) % len(payloads)] for i in range(b)] for k in range(WARMUP + SAMPLES)]
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
        out = []
        for k, batch in enumerate(batches):
            torch.cuda.synchronize(); t0 = time.perf_counter()
            codec.decode_batch(batch)
            torch.cuda.synchronize(); dt = time.perf_counter() - t0
            if k >= WARMUP:
                out.append(dt)
        samples[str(b)] = out; vram[str(b)] = int(torch.cuda.max_memory_allocated())
        print("B=%d mean %.1f ms" % (b, 1e3 * float(np.mean(out))))
    res = {"samples": samples, "emulated_batching": False, "m": M, "peak_vram_bytes": vram,
           "n_payloads": len(payloads), "gpu": torch.cuda.get_device_name(0), "git_sha": GIT_SHA}
    Path("/kaggle/working/latency_v1_true.json").write_text(json.dumps(res))
    print("wrote latency_v1_true.json")
except Exception:
    Path("/kaggle/working/latency_v1_true.json").write_text(json.dumps({"error": traceback.format_exc(), "emulated_batching": False}))
    raise

In [ ]:
import datetime, json, os, platform, subprocess, time
STEP, ARGS = "edge-v1-latency", []
def _gpu():
    try:
        return subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip()
    except Exception:
        return None
try:
    import torch; torch_version = torch.__version__
except Exception:
    torch_version = None
out = "/kaggle/working/runs/" + STEP
os.makedirs(out, exist_ok=True)
manifest = {"step": STEP, "entrypoint": "Codec.edge.sqgan_profile", "args": ARGS, "git_sha": GIT_SHA,
            "utc": datetime.datetime.now(datetime.timezone.utc).isoformat(), "python": platform.python_version(),
            "torch": torch_version, "gpu": _gpu(), "wall_time_s": time.time() - T0}
json.dump(manifest, open(out + "/run_manifest.json", "w"), indent=2)
print(manifest)